In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold

from lightgbm import LGBMClassifier
from xgboost import  XGBClassifier
from sklearn.ensemble import  RandomForestClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import f1_score


import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Task 1: Write your code here:

food_delivery_path = os.path.join(path, 'Q1_data.csv')
df_food_delivery = pd.read_csv(food_delivery_path)


In [ ]:
# Task 2: Write your code here:
print(f"Shape: {df_food_delivery.shape}")
df_food_delivery.head()


In [ ]:
# Task 3: Write your code here:
# Check data info
df_food_delivery.info()

In [ ]:
# Task 4: Write your code here:
df_food_delivery.describe()

In [ ]:
#  distribution (target variable)


In [ ]:
# Task 5: Write your code here:
# Target distribution

#df_food_delivery

# Target distribution
print(f"Legendary: {df_food_delivery['is_legendary'].sum()}")
print(f"Normal: {(df_food_delivery['is_legendary'] == 0).sum()}")

In [ ]:
def check_target_distribution(df, target_column):
  df_food_delivery[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("delivery_time")
  plt.grid(False)

  plt.show()

check_target_distribution(df_food_delivery, "delivery_time")

In [ ]:
# Drop ID column
data = df_food_delivery.drop('Order_ID', axis=1)

In [ ]:
# Task 1: Write your code here:
#df_food_delivery
# Select columns
cols = ['Order_ID', 'Weather', 'Time_of_Day', 'Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs', 'Delivery_Time', 'Distance_km', 'Traffic_Level',]
df_clean = df_food_delivery[cols].copy()
#df.drop(columns=['Order_ID])
# Drop rows where target (Order_ID)
print(f"Before: {df_clean.shape}")
df_food_delivery.drop(columns=['Order_ID'])
df_clean = df_clean.dropna(subset=['Order_ID',])
print(f"After dropping missing Order_ID: {df_clean.shape}")

In [ ]:
# Task 2: Write your code here:
# Check missing values
missing_values = data.isnull().sum()
print("Columns with missing values:")
print(missing_values[missing_values > 0])

# Fill missing values with mean for each column
data = data.fillna(data.mean())

In [ ]:
# Task 3: Write your code here:
# 4. Do we have duplicate samples?
def check_duplicates(df):
    duplicates = df.duplicated().sum()
    print(f"Number of Duplicate Samples: {duplicates}")
    if duplicates > 0:
        print("Dropping Duplicates...")
        df.drop_duplicates(inplace=True)
        print("Duplicates Dropped.")
    else:
        print("No Duplicate Samples Found.")

check_duplicates(train_df)

In [ ]:
# Task 4: Write your code here:
# Encode categorical columns - converts text to integers
categorical_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Time_of_Day', 'Vehicle_Type', ]
for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

df_clean.head()

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import OneHotEncoder #import OneHotEncoder

print('data before encoding:\n', categorical_cols) #show before encoding

onehot_encoder = OneHotEncoder(sparse_output=False) # Instantiate OneHotEncoder
data_onehot_encoded = onehot_encoder.fit_transform(categorical_cols) # Apply fit_transform to the copied

print('\nData after encoding:\n', data_onehot_encoded) #show after encoding

In [ ]:
# Task 6: Write your code here:

In [ ]:
# Task 1: Write your code here:
# Define features (X) and target (y)
feature_cols = cols = ['Order_ID', 'Weather', 'Time_of_Day', 'Vehicle_Type', 'Preparation_Time_min',
                       'Courier_Experience_yrs', 'Delivery_Time', 'Distance_km', 'Traffic_Level',]
X = df_clean[feature_cols]
y = df_clean['Order_ID']

# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")



In [ ]:
print(f"\nFeature ranges - Min: {X_train.min().min():.2f}, Max: {X_train.max().max():.2f}")
X_train.head(3)

In [ ]:
# Task 2,3,4,5: Write your code here:
# Define classification models
models = {

    "Random Forest Classifier": RandomForestClassifier(n_estimators=100),

}

models["Ensemble"] = VotingClassifier(estimators=[
        ('lr', models[('gnb', models["Random Forest Classifier"])], voting="soft") ]

for model_name, model in models.items():
    scores_accuracy = []
    scores_precision = []
    scores_recall = []
    scores_f1 = []

    # Stratified 5-Fold Cross-Validation
    skf = KFold(n_splits=5)
    for train_index, test_index in skf.split(X, y):

        X_Train, X_Test = X.loc[train_index, :], X.loc[test_index, :]
        y_Train, y_Test = y.iloc[train_index], y.iloc[test_index]

        # Train the model
        model.fit(X_Train, y_Train)

        # Predict on the test set
        y_pred = model.predict(X_Test)

        # Calculate metrics
        scores_f1.append(f1_score(y_Test, y_pred, average='weighted'))




MAE = data['emission'].median()
MAE = mean_absolute_error(data['emission'], [MAE] * len(data))
print(f"MAE: {mae}")


    print(f"{model_name} F1-Score: {np.mean(scores_f1):.4f}")
    print("\n")



In [ ]:
# Task 1: Write your code here:

# Plot feature importances
plt.figure(figsize=(18, 14))
plt.barh(features, importances, color='orange')
plt.xlabel('Preparation_Time_min')
plt.ylabel('delivery_time')
plt.title('Random Forest Classifier')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

def check_target_distribution(df, target_column):
  df_food_delivery[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("delivery_time")
  plt.grid(False)

  plt.show()

check_target_distribution(df_food_delivery, "delivery_time")


In [ ]:
# Task Bonus: Write your code here:

models = {

    "Random Forest Classifier": RandomForestClassifier(n_estimators=100),
    "CatBoost Classifier": CatBoostClassifier(verbose=0)
}

models["Ensemble"] = VotingClassifier(estimators=[
        ('lr', models["CatBoost Classifier"]),  ('gnb', models["Random Forest Classifier"])], voting="soft")

for model_name, model in models.items():
    scores_accuracy = []
    scores_precision = []
    scores_recall = []
    scores_f1 = []


    skf = KFold(n_splits=5)
    for train_index, test_index in skf.split(X, y):

        X_Train, X_Test = X.loc[train_index, :], X.loc[test_index, :]
        y_Train, y_Test = y.iloc[train_index], y.iloc[test_index]
        # Train the model
        model.fit(X_Train, y_Train)

        y_pred = model.predict(X_Test)

        # Calculate metrics
        scores_f1.append(f1_score(y_Test, y_pred, average='weighted'))

    # Print the results
    print(f"{model_name} F1-Score: {np.mean(scores_f1):.4f}")
    print("\n")
